In [ ]:
# utils.py

import os
from datetime import datetime, date
from dotenv import load_dotenv

load_dotenv()

GOOGLE_SHEETS_ID = os.getenv("GOOGLE_SHEETS_ID")

def get_today_str_iso():
    """YYYY-MM-DD, para registro de data completo (não usado na célula)"""
    return date.today().isoformat()

def get_today_day():
    """Retorna apenas o dia do mês (1 a 31)."""
    return datetime.today().day

def get_columns_for_today():
    """
    Calcula colunas da estrutura única de planilha.

    Estrutura (linha 2):
    A: mês JAN,   A2: "Data"  B2:"Entrada" C2:"Saída" D2:"Diário" E2:"Saldo" F2:""
    G: mês FEV,   G2: "Data"  H2:"Entrada" I2:"Saída" J2:"Diário" K2:"Saldo" L2:""
    M: mês MAR, ...

    Cada bloco de mês tem 6 colunas (Data, Entrada, Saída, Diário, Saldo, Vazia).
    Índices 1-based (A=1, B=2, ...).
    """
    # offset em colunas por mês (0, 6, 12, 18, ...)
    meses_offset = {
        1: 0,   # JAN
        2: 6,   # FEV
        3: 12,  # MAR
        4: 18,  # ABR
        5: 24,  # MAI
        6: 30,  # JUN
        7: 36,  # JUL
        8: 42,  # AGO
        9: 48,  # SET
        10:54,  # OUT
        11:60,  # NOV
        12:66   # DEZ
    }
    hoje = datetime.now()
    mes_offset = meses_offset[hoje.month]
    dia = hoje.day
    row = 2 + dia  # linha 3 para dia 1, linha 4 para dia 2, ...

    data_col = 1 + mes_offset
    entrada_col = 2 + mes_offset
    saida_col = 3 + mes_offset
    diario_col = 4 + mes_offset
    saldo_col = 5 + mes_offset

    return {
        "row": row,
        "data_col": data_col,
        "entrada_col": entrada_col,
        "saida_col": saida_col,
        "diario_col": diario_col,
        "saldo_col": saldo_col
    }

def get_sheet_name():
    """Nome da aba é o ano atual, ex: "2024"."""
    return str(datetime.now().year)

In [ ]:
print(f"Today's date (ISO): {get_today_str_iso()}")
print(f"Today's day: {get_today_day()}")
print(f"Today's columns: {get_columns_for_today()}")

Today's date (ISO): 2026-02-18
Today's day: 18
Today's columns: {'row': 20, 'data_col': 7, 'entrada_col': 8, 'saida_col': 9, 'diario_col': 10, 'saldo_col': 11}


In [ ]:
# gemini_parser.py

from google import genai
import json
import os
from dotenv import load_dotenv
# from utils import get_today_str_iso

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

MODEL_NAME = "gemini-1.5-flash"

def parse_audio_expense(audio_path: str):
    """
    Transcreve o áudio e retorna um dict:
    {
      "tipo": "receita" | "despesa_fixa" | "despesa_diaria",
      "valor": float,
      "categoria": str,
      "data": "YYYY-MM-DD",
      "descricao": str
    }
    """
    # instruções claras para o modelo
    prompt = f"""
    Você é um assistente financeiro que classifica gastos e receitas.

    REGRAS PARA TIPOS:
    - "receita": dinheiro que entra (salário, reembolso, rendimentos, etc.).
    - "despesa_fixa": contas mensais ou recorrentes (energia, água, gás, condomínio, aluguel, wifi, telefone, cartão de crédito, seguro, etc.).
    - "despesa_diaria": gastos do dia a dia (mercado, restaurante, lanchonete, combustível, transporte, farmácia, bar, lazer, etc.).

    Exemplos:
    - "Gastei 50 reais de energia" -> tipo = "despesa_fixa"
    - "Paguei o condomínio hoje" -> tipo = "despesa_fixa"
    - "Gastei 20 reais no mercado" -> tipo = "despesa_diaria"
    - "Comprei um lanche de 15" -> tipo = "despesa_diaria"
    - "Recebi 500 de salário" -> tipo = "receita"

    CAMPO DATA:
    - Se o áudio falar "hoje", "agora" ou não falar data, use "{get_today_str_iso()}".
    - Se mencionar explicitamente uma data (ex: "dia 10", "10 de fevereiro"), converta para o formato "YYYY-MM-DD" correto.

    FORMATO DE RESPOSTA:
    Responda APENAS com um JSON válido, sem texto extra, no formato:

    {{
      "tipo": "receita" | "despesa_fixa" | "despesa_diaria",
      "valor": 13.8,  // use ponto como separador decimal
      "categoria": "mercado",
      "data": "YYYY-MM-DD",
      "descricao": "Gasto no mercado"
    }}
    """
    audio_file = client.files.upload(audio_path)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[prompt, audio_file],
    )
    try:
        text = response.text.strip()